In [1]:
!pip install -U gensim

In [9]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import re

# --- 1. 코퍼스 구성 (20개 문장) ---
corpus = [
    "팔십하고도 나흘이 지나도록 그는 고기를 한 마리도 잡지 못했다",
    "사람은 파멸당할 수는 있을지 몰라도 패배는 하지 않는다",
    "오늘 나는 학교에 가지 않았다",
    "차 두 대가 빨간 불에 걸리지 않으려고 가속으로 내달았다",
    "멀리 보이는 배들에는 모든 사람들의 소원이 실려 있다",
    "행복한 가정은 모두 비슷하게 닮았지만, 불행한 가정은 저마다의 이유로 불행하다",
    "꽃은 자신이 직접 사겠노라고 댈러웨이 부인은 말했다",
    "우리는 지금 전방에서 9킬로미터 정도 떨어진 곳에 있다",
    "무사태평하게 보이는 사람들도 마음속 깊은 곳을 두드려보면 어딘가 슬픈 소리가 난다",
    "때로는 크리스마스에도 악마 같은 아이가 태어난다",
    "4월의 맑고 쌀쌀한 어느 날, 시계가 13번 울리고 있었다",
    "그는 빅 브라더를 사랑했다",
    "국경의 긴 터널을 빠져나오자, 설국이였다",
    "이렇게 슬픈 이야기는 들어본 적이 없다",
    "사람들은 아버지를 난장이라고 불렀다",
    "부끄러움 많은 생애를 보냈습니다",
    "모든 동물은 평등하다",
    "땅 속 어느 굴에 한 호빗이 살고 있었다",
    "독서는 지식을 넓히고 상상력을 키워준다",
    "최근에 흥미로운 소설책 한 권을 다 읽었다"
]

# --- 2. 텍스트 전처리 ---
def preprocess(text):
    text = re.sub(r'[^가-힣\s]', '', text)
    return text

tokenized_corpus = [preprocess(doc).split() for doc in corpus]

# --- 3. Word2Vec 모델 학습 ---
model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,  # 임베딩 벡터의 차원
    window=5,         # 컨텍스트 윈도우 크기
    min_count=1,      # 단어의 최소 빈도수
    workers=10,        # 학습을 위한 프로세스 수
    sg=1              # 0: CBOW, 1: Skip-gram
)

print("Word2Vec 모델 학습 완료!")

# --- 4. 문장 임베딩 (단어 벡터의 평균 계산) [cite: 671] ---
sentence_embeddings = []
for tokens in tokenized_corpus:
    # 문장에 있는 단어 중 모델의 어휘 사전에 있는 단어만 필터링
    valid_words = [word for word in tokens if word in model.wv]

    if not valid_words:
        # 유효한 단어가 없는 경우, 0벡터로 처리
        embedding = np.zeros(model.vector_size, dtype=np.float32)
    else:
        # 단어 벡터들의 합을 구한 뒤, 단어 개수로 나누어 평균 계산
        embedding = np.mean([model.wv[word] for word in valid_words], axis=0)

    sentence_embeddings.append(embedding)

# 리스트를 NumPy 배열로 변환
sentence_embeddings = np.array(sentence_embeddings)

print(f"\n총 {len(sentence_embeddings)}개의 문장이 벡터로 변환되었습니다.")
print(f"변환된 벡터의 형태: {sentence_embeddings.shape}")

# --- 5. 문장 간 코사인 유사도 계산 [cite: 673] ---
# 모든 문장 쌍 간의 코사인 유사도를 한 번에 계산
cosine_sim_matrix = cosine_similarity(sentence_embeddings)

# --- 6. 가장 유사한/관계없는 문장 쌍 찾기
# 행렬의 크기 (문장 개수)
n = len(corpus)

# 2차원 유사도 행렬을 1차원 배열로 변환
flat_sim = cosine_sim_matrix.flatten()

# 유사도 점수를 기준으로 인덱스를 정렬 (오름차순)
# argsort()는 값 자체 대신, 정렬된 순서의 '인덱스'를 반환합니다.
sorted_indices = np.argsort(flat_sim)

# --- 가장 관계없는 문장 쌍 찾기 ---
# 정렬된 인덱스 중 가장 첫 번째(가장 작은 값)의 인덱스를 가져옴
min_flat_idx = sorted_indices[0]
# 1차원 인덱스를 다시 2차원 행렬의 (행, 열) 인덱스로 변환
min_sim_idx = np.unravel_index(min_flat_idx, cosine_sim_matrix.shape)
min_sim_score = cosine_sim_matrix[min_sim_idx]

# --- 가장 유사한 문장 쌍 찾기 ---
# 가장 큰 값들은 자기 자신을 비교한 대각선 값(1.0)들이 차지함 (총 n개)
# 따라서 그 바로 앞 인덱스(n+1번째)가 실제 가장 유사한 문장 쌍의 인덱스가 됨
max_flat_idx = sorted_indices[-(n + 1)]
# 1차원 인덱스를 다시 2차원 행렬의 (행, 열) 인덱스로 변환
max_sim_idx = np.unravel_index(max_flat_idx, cosine_sim_matrix.shape)
max_sim_score = cosine_sim_matrix[max_sim_idx]

print("\n--- 결과 분석 ---")

# 가장 유사한 문장 쌍 출력
print("\n가장 유사한 문장 쌍:")
print(f"   - 문장 {max_sim_idx[0]+1}: \"{corpus[max_sim_idx[0]]}\"")
print(f"   - 문장 {max_sim_idx[1]+1}: \"{corpus[max_sim_idx[1]]}\"")
print(f"   - 유사도 점수: {max_sim_score:.4f}")

# 가장 관계없는 문장 쌍 출력
print("\n 가장 관계없는 문장 쌍:")
print(f"   - 문장 {min_sim_idx[0]+1}: \"{corpus[min_sim_idx[0]]}\"")
print(f"   - 문장 {min_sim_idx[1]+1}: \"{corpus[min_sim_idx[1]]}\"")
print(f"   - 유사도 점수: {min_sim_score:.4f}")

Word2Vec 모델 학습 완료!

총 20개의 문장이 벡터로 변환되었습니다.
변환된 벡터의 형태: (20, 100)

--- 결과 분석 ---

가장 유사한 문장 쌍:
   - 문장 20: "최근에 흥미로운 소설책 한 권을 다 읽었다"
   - 문장 18: "땅 속 어느 굴에 한 호빗이 살고 있었다"
   - 유사도 점수: 0.2724

 가장 관계없는 문장 쌍:
   - 문장 9: "무사태평하게 보이는 사람들도 마음속 깊은 곳을 두드려보면 어딘가 슬픈 소리가 난다"
   - 문장 15: "사람들은 아버지를 난장이라고 불렀다"
   - 유사도 점수: -0.2831
